# Reported Crime Incidents By Year

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/vancouver_crime_clean.csv")

df.head()

In [ ]:
crime_by_year = df.groupby("year").size()
crime_by_year

In [ ]:
crime_by_type = df["type"].value_counts()
crime_by_type

In [ ]:
crime_by_year = df.groupby("year").size()

plt.figure(figsize = (10, 5))
crime_by_year.plot(kind = "bar")
plt.title("Reported Crime Incidents by Year in Vancouver")
plt.xlabel("Year")
plt.ylabel("Number of Reported Incidents")
plt.tight_layout()
plt.savefig("../results/figures/incidents_by_year.png", dpi = 300)
plt.show()

In [ ]:
top_neighbourhoods = df["neighbourhood"].value_counts().head(10)

plt.figure(figsize = (10, 5))
top_neighbourhoods.plot(kind = "bar")
plt.title("Top 10 Neighbourhoods by Reported Incidents")
plt.xlabel("Neighbourhood")
plt.ylabel("Number of Reported Incidents")
plt.xticks(rotation = 45, ha = "right")
plt.tight_layout()
plt.savefig("../results/figures/top_neighbourhoods.png", dpi = 300)
plt.show()

Top 10 Neighbourhoods & Their Crime Composition

In [ ]:
top10_df = df[df["neighbourhood"].isin(top_neighbourhoods.index)].copy()

In [ ]:
# crime table of type x neighbourhood
crime_table = (
    top10_df
    .groupby(["neighbourhood", "type"])
    .size()
    .unstack(fill_value = 0)
)

# adds total and sorts
crime_table["total"] = crime_table.sum(axis=1)
crime_table = crime_table.sort_values("total", ascending=True)

crime_table.head()

In [ ]:
# keeps top 5 crime types and reduce the rest into "Other"
type_only = crime_table.drop(columns = "total") # (drop total column to do this)
top_types = type_only.sum(axis = 0).sort_values(ascending = False).head(5).index

crime_table_reduced = type_only[top_types].copy()
crime_table_reduced["Other"] = type_only.drop(columns = top_types).sum(axis = 1)
crime_table_reduced["total"] = crime_table_reduced.sum(axis = 1)
crime_table_reduced.head()

In [ ]:
# plotting data
plot_data = crime_table_reduced.drop(columns = "total")

fig, ax = plt.subplots(figsize = (12, 8))

plot_data.plot(
    kind = "barh",
    stacked = True,
    ax = ax
)

ax.set_title("Crime Composition in the Top 10 Vancouver Neighbourhoods", fontsize = 14)
ax.set_xlabel("Number of Reported Incidents")
ax.set_ylabel("Neighbourhood")

for i, total in enumerate(crime_table_reduced["total"]):
    ax.text(total + 5, i, str(total), va = "center", fontsize=9)

plt.legend(title = "Crime Type", bbox_to_anchor=(1.05, 1), loc = "upper left")
plt.tight_layout()
plt.savefig("../results/figures/top10_neighbourhood_crime_composition.png", dpi = 300, bbox_inches = "tight")
plt.show()

Analysis:
Common crimes for all top 10 most-crime-occurring neighbourhoods seem to be "Theft from Vehicle" or just "Other Theft". To note, although it is less than the top 2, "Offence Against A Person" is still one of the 5 most prevalently occurring crimes in these neighbourhoods.

Following question: In each neighbourhood, what percentage of reported incidents are each crime type?

In [ ]:
# getting percentage
composition = crime_table_reduced.drop(columns = "total").div(
    crime_table_reduced["total"],
    axis = 0
) * 100

fig, ax = plt.subplots(figsize = (12, 8))

composition.plot(
    kind = "barh",
    stacked = True,
    ax = ax
)

# plot
ax.set_title("Crime Type Composition in the Top 10 Vancouver Neighbourhoods", fontsize = 14)
ax.set_xlabel("Percentage of Reported Incidents")
ax.set_ylabel("Neighbourhood")

plt.legend(title = "Crime Type", bbox_to_anchor = (1.05, 1), loc = "upper left")
plt.tight_layout()
plt.savefig("../results/figures/top10_neighbourhood_crime_composition_percentage.png", dpi = 300, bbox_inches = "tight")
plt.show()